### 01. First Test: Capturing and Inspecting Raw Telemetry

This initial cell acts as a sanity check. It connects to the `raw.vehicle-positions` Kafka topic as a temporary consumer and pulls a finite batch of 500 real messages. 

The goal here is not to compute analytics yet, but to observe the raw shape, noise, and cadence of the MBTA data. I extracted the GeoJSON `location` coordinates into flat `lat` and `lon` columns, loaded the batch into a `pandas` DataFrame, and used DuckDB to sort the results chronologically per vehicle. This gives us our first raw glimpse into a vehicle's breadcrumb trail.

In [19]:
import json
import time
import pandas as pd
import duckdb
from confluent_kafka import Consumer, KafkaException

conf = {
    'bootstrap.servers': '127.0.0.1:9092',
    'group.id': f'duckdb-explorer-{int(time.time())}',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
}

consumer = Consumer(conf)
topic = 'raw.vehicle-positions'
consumer.subscribe([topic])

print(f"Collecting data from topic '{topic}'...")

messages = []
TARGET = 20000
MAX_WAIT_SECONDS = 30
start = time.time()

try:
    while len(messages) < TARGET and (time.time() - start) < MAX_WAIT_SECONDS:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            raise KafkaException(msg.error())

        payload = json.loads(msg.value().decode('utf-8'))

        if 'location' in payload and 'coordinates' in payload['location']:
            payload['lon'] = payload['location']['coordinates'][0]
            payload['lat'] = payload['location']['coordinates'][1]

        messages.append(payload)

except KeyboardInterrupt:
    print("Capture manually stoped")
finally:
    consumer.close()

df_analitics = pd.DataFrame(messages)
print(f"\n{len(df_analitics)} pings stored in Pandas")

query = """
    SELECT 
        vehicle_id,
        trip_id,
        route_id,
        timestamp,
        lat,
        lon
    FROM df_analitics
    ORDER BY vehicle_id, timestamp
"""

df_analitics = duckdb.sql(query).df()
df_analitics.head(10)


20000 pings stored in Pandas


,vehicle_id,trip_id,route_id,timestamp,lat,lon
0,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:06:12.000Z,42.190735,-70.739998
1,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:06:12.000Z,42.190735,-70.739998
2,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:06:43.000Z,42.197193,-70.741241
3,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:06:43.000Z,42.197193,-70.741241
4,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:07:14.000Z,42.202873,-70.745308
5,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:07:14.000Z,42.202873,-70.745308
6,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:07:45.000Z,42.206917,-70.752884
7,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:07:45.000Z,42.206917,-70.752884
8,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:08:16.000Z,42.211399,-70.760658
9,1634,SouthBase-793230-6082,CR-Greenbush,2026-08-15T19:08:47.000Z,42.215149,-70.769180


### 02. Second Test: Deduplication and Real Update Frequency (Gaps)

After inspecting the initial raw batch, I noticed duplicate consecutive pings for the same vehicle (identical timestamps and coordinates). This happens because our ingestion service polls the MBTA feed (every 10-15 seconds) faster than the actual vehicles update their internal GPS.

To compute accurate metrics later on, we must filter out this noise. This cell uses DuckDB to deduplicate consecutive identical records. Then, it applies the `LAG()` window function to calculate the actual time gap (in seconds) between genuine vehicle movements, revealing the true update frequency of our telemetry stream.

In [21]:
query_gaps = """
    WITH deduplicated_pings AS (
        -- Deleting identical records
        SELECT 
            vehicle_id,
            trip_id,
            route_id,
            CAST(timestamp AS TIMESTAMPTZ) AS timestamp,
            lat,
            lon
        FROM df_analitics
        GROUP BY vehicle_id, trip_id, route_id, timestamp, lat, lon
    )
    -- Calculate the diff between pings for every vehicle
    SELECT 
        vehicle_id,
        timestamp,
        trip_id,
        route_id,
        lat,
        lon,
        LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp) AS prev_timestamp,
        date_diff('second', 
            LAG(timestamp) OVER (PARTITION BY vehicle_id ORDER BY timestamp), 
            timestamp
        ) AS real_update_gap_seconds
    FROM deduplicated_pings
    ORDER BY vehicle_id, timestamp
"""

df_gaps = duckdb.sql(query_gaps).df()
display(df_gaps.head(15))

,vehicle_id,timestamp,trip_id,route_id,lat,lon,prev_timestamp,real_update_gap_seconds
0,1634,2026-08-15 13:06:12-06:00,SouthBase-793230-6082,CR-Greenbush,42.190735,-70.739998,NaT,<NA>
1,1634,2026-08-15 13:06:43-06:00,SouthBase-793230-6082,CR-Greenbush,42.197193,-70.741241,2026-08-15 13:06:12-06:00,31
2,1634,2026-08-15 13:07:14-06:00,SouthBase-793230-6082,CR-Greenbush,42.202873,-70.745308,2026-08-15 13:06:43-06:00,31
3,1634,2026-08-15 13:07:45-06:00,SouthBase-793230-6082,CR-Greenbush,42.206917,-70.752884,2026-08-15 13:07:14-06:00,31
4,1634,2026-08-15 13:08:16-06:00,SouthBase-793230-6082,CR-Greenbush,42.211399,-70.760658,2026-08-15 13:07:45-06:00,31
5,1634,2026-08-15 13:08:47-06:00,SouthBase-793230-6082,CR-Greenbush,42.215149,-70.769180,2026-08-15 13:08:16-06:00,31
6,1634,2026-08-15 13:09:18-06:00,SouthBase-793230-6082,CR-Greenbush,42.217548,-70.777443,2026-08-15 13:08:47-06:00,31
7,1634,2026-08-15 13:09:49-06:00,SouthBase-793230-6082,CR-Greenbush,42.218521,-70.783806,2026-08-15 13:09:18-06:00,31
8,1634,2026-08-15 13:10:20-06:00,SouthBase-793230-6082,CR-Greenbush,42.219448,-70.788506,2026-08-15 13:09:49-06:00,31
9,1634,2026-08-15 13:10:51-06:00,SouthBase-793230-6082,CR-Greenbush,42.219879,-70.789406,2026-08-15 13:10:20-06:00,31


#### 02.5. Gap Distribution & Silent Vehicle Thresholds

To build a reliable "stale vehicle" detection system, it must be established a baseline for normal update frequencies. This cell calculates the summary statistics and quantiles for `real_update_gap_seconds` metric. 

The data reveals that while the median update gap is 16 seconds, the 99th percentile sits at 76 seconds. This provides a data-driven threshold: if a vehicle goes silent for more than ~90-120 seconds, we can confidently flag it as a feed interruption, a "silent vehicle", or an out-of-service anomaly, rather than normal network jitter.

In [6]:
df_gaps['real_update_gap_seconds'].describe()

count       7712.0
mean     19.034362
std      24.454468
min            0.0
25%           12.0
50%           16.0
75%           19.0
max          939.0
Name: real_update_gap_seconds, dtype: Float64

In [7]:
df_gaps['real_update_gap_seconds'].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

0.50    16
0.75    19
0.90    29
0.95    34
0.99    69
Name: real_update_gap_seconds, dtype: Int64

### 03. Third Test: Spatial Map Matching against GTFS-Static

Now that we have a clean, deduplicated sequence of vehicle pings, we need to anchor these raw GPS coordinates to the physical transit network. A raw latitude/longitude pair is much more useful once we can relate it to the sequence of stops defined by the vehicle's current GTFS trip.

This cell performs a spatial join between our real-time telemetry (`df_gaps`) and the MBTA's GTFS-static schedule (`stop_times.txt`, `stops.txt`, and `routes.txt`). For each telemetry ping, we retrieve the stops belonging to its specific `trip_id` and calculate the geographic distance between the vehicle and each of those candidate stops. 

At this stage, however, we **do not select the closest stop yet**. The previous implementation used a `QUALIFY ROW_NUMBER()` clause to immediately keep only the single closest stop for each `(vehicle_id, timestamp)`:

```sql
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY vehicle_id, timestamp
    ORDER BY dist_sq_meters ASC
) = 1
```

This approach worked as a basic nearest-neighbor lookup, but it introduced an important limitation: **it considered each GPS ping independently**. The selected stop was determined solely by physical distance, with no knowledge of which stop had been matched in the previous ping.

That was creating a potential problem because GTFS provides an ordered `stop_sequence` for every trip. A vehicle traveling along a trip should generally progress through that sequence rather than jumping arbitrarily between stops based only on which one happens to be geographically closest ($stop_sequence[t] >= stop_sequence[t-1]$). For example, if a vehicle is physically closer to stop 20 than stop 10, that does not necessarily mean it has already reached stop 20 if the previous ping was still associated with stop 10, because the vehicle could be at the stop 10 for x seconds picking up passengers.

For that reason, the `QUALIFY` step was removed from this stage. The query now returns **all candidate stops and their corresponding distances for every GPS ping**. This preserves the information needed for the next stage, where the matching process will become sequential rather than purely spatial.

The new flow is therefore:

**GPS ping → candidate stops for the trip → distance calculation → sequence-aware matching**

### Geospatial Correction & Metric Conversion

Originally, I was using a calculation based directly on latitude and longitude differences:

$$
(\Delta \text{lat})^2 + (\Delta \text{lon})^2
$$

This had an important problem: differences in latitude and longitude are expressed in degrees, but a degree of latitude and a degree of longitude do not represent the same physical distance.

To convert the coordinates into an approximate local metric system, I use the Equirectangular approximation:

$$
d =
\sqrt{
(\Delta \text{lat} \times 111320)^2 +
(\Delta \text{lon} \times 111320 \times \cos(\text{lat}))^2
}
$$

where `111320` approximates the number of meters represented by one degree of latitude.

The cosine correction is applied to longitude because the physical length represented by one degree of longitude decreases with latitude. This produces a distance expressed directly in **meters**, which is much more useful for the proximity and bunching analysis that will follow.The resulting `distance_meters` column therefore represents an approximate physical distance between each vehicle ping and each candidate GTFS stop.

At this stage, a vehicle can have several candidate rows for the same timestamp:

```text
vehicle   timestamp   stop             sequence   distance_meters
1634      13:10:20    Greenbush             0         1439.606
1634      13:10:20    North Scituate       10           11.931
1634      13:10:20    Cohasset             20         1842.552
...
```

This is intentional, since we now retain the complete candidate set so that the next step can combine **spatial proximity** with the **known order of stops** rather than blindly selecting the nearest geographic point.

*Note: Real-world data is messy. During the initial processing, DuckDB produced a `Conversion Error` because it inferred `trip_id` as a numeric type. The MBTA data also contains custom alphanumeric trip IDs, such as `8pmChrisBrownUsher-847359-4763`, created for additional service. To prevent type inference from breaking the query, the relevant GTFS identifiers are explicitly loaded as `VARCHAR` using DuckDB's `types` parameter.*

In [23]:
# Next step: Spatial Map Matching with Routes Names
# Comparing data of the current position vs the strict sequence of the vehicle stops

query_stops_candidates = """
    WITH stops_for_trip AS (
        SELECT 
            p.vehicle_id,
            p.trip_id,
            p.timestamp,
            p.lat AS ping_lat,
            p.lon AS ping_lon,
            
            r.route_short_name,
            r.route_long_name,
            
            st.stop_sequence,
            s.stop_id,
            s.stop_name,
            
            -- Approximate meters-per-degree at Boston's latitude, corrected for longitude compression
            (
                POW(
                (p.lat - s.stop_lat) * 111320, 
                2
                ) +
                POW(
                (p.lon - s.stop_lon) * 111320 * COS(RADIANS(p.lat)), 
                2
                )
            ) AS dist_sq_meters
        
        FROM df_gaps AS p
        
        -- Forcing DuckDB to read the IDs as VARCHAR to avoid failing at alphanumeric IDs
        JOIN read_csv_auto(
        '../gtfs_static/MBTA_GTFS/stop_times.txt', 
        types={'trip_id': 'VARCHAR', 'stop_id': 'VARCHAR'}
        ) AS st 
            ON p.trip_id = st.trip_id
        JOIN read_csv_auto(
        '../gtfs_static/MBTA_GTFS/stops.txt', 
        types={'stop_id': 'VARCHAR'}
        ) AS s 
            ON st.stop_id = s.stop_id
        JOIN read_csv_auto(
        '../gtfs_static/MBTA_GTFS/routes.txt', 
        types={'route_id': 'VARCHAR'}
        ) AS r
            ON p.route_id = r.route_id
    ORDER BY
        p.vehicle_id,
        p.trip_id,
        p.timestamp,
        st.stop_sequence 
"""

df_candidates = duckdb.sql(query_map_matching).df()
display(df_candidates.head(20))

,vehicle_id,route_short_name,route_long_name,trip_id,timestamp,closest_stop,stop_sequence,distance_meters
0,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:06:12-06:00,Greenbush,0,1439.606
1,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:06:43-06:00,Greenbush,0,2097.999
2,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:07:14-06:00,Greenbush,0,2684.755
3,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:07:45-06:00,Greenbush,0,3174.649
4,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:08:16-06:00,North Scituate,10,2475.337
5,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:08:47-06:00,North Scituate,10,1673.812
6,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:09:18-06:00,North Scituate,10,946.018
7,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:09:49-06:00,North Scituate,10,410.985
8,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:10:20-06:00,North Scituate,10,11.931
9,1634,NaN,Greenbush Line,SouthBase-793230-6082,2026-08-15 13:10:51-06:00,North Scituate,10,76.938
